# Spec-Craft Quickstart: Automated Spec-Driven Development with aichat & llm-functions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akuroiwa/spec-craft/blob/main/docs/source/notebooks/spec_craft_quickstart.ipynb)

Welcome to the **Spec-Craft** automated batch execution tutorial on Google Colab!

**Spec-Craft** is an Obsidian-based Spec-Driven Development (SDD) toolkit designed for AI agents and developers. It establishes a clean separation of concerns:
- **Strategy (Obsidian)**: Defines the *What* and *Why* (the project's strategic roadmap and domain model).
- **Tactics (spec-kit & code)**: Executes the *How* (iterative implementation cycles, TDD, and test suites).

### Architecture: aichat, llm-functions, & spec-craft MCP Bridge
In `sigoden/aichat`, tool calling and MCP server integration are facilitated through **`sigoden/llm-functions`** and the **`argc` MCP Bridge**:

```mermaid
flowchart LR
    A[aichat CLI] -->|function calling| B[llm-functions]
    B --> C[Built-in Tools]
    B -->|argc mcp bridge| D[mcp.json]
    D --> E[spec-craft FastMCP Server]
```

> **Workflow Note**:
> This notebook demonstrates **automated batch execution** using `sigoden/aichat` as a command-line tool.
> For **interactive development with an autonomous AI coding agent** using native MCP and live terminal TUI, see [spec_craft_opencode.ipynb](https://colab.research.google.com/github/akuroiwa/spec-craft/blob/main/docs/source/notebooks/spec_craft_opencode.ipynb).

In this notebook, you will learn how to:
1. Set up a local LLM environment using **Ollama** (Gemma 4 QAT with extended context `OLLAMA_CONTEXT_LENGTH=32768`), **sigoden/aichat** (with Rust/Cargo), **argc**, and **sigoden/llm-functions**.
2. Configure and start the **`llm-functions` MCP Bridge** pointing to `spec-craft` (`uv run spec-craft serve`).
3. Generate an Obsidian strategic feature specification (`obsidian/strategy_task_tracker.md`) automatically.
4. Inspect and parse the strategic roadmap using **Spec-Craft**'s `StoryboardParser`.
5. Automatically generate modular Python implementation (`task_tracker.py`) and focused unit tests (`test_task_tracker.py`).
6. Verify behavioral correctness by running tests directly in Colab.


In [ ]:
# ==============================================================================
# 1. Environment Setup (Rust, aichat, argc, llm-functions, Ollama, specify-cli, spec-craft)
# ==============================================================================
import os, sys, time, subprocess

print("📦 1. Installing system dependencies & Rust toolchain...")
!apt-get update -qq && apt-get install -y -qq build-essential curl zstd git pkg-config libssl-dev > /dev/null

# Rust / Cargo setup (https://rust-lang.org/ja/tools/install/)
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ["PATH"] = f"{os.path.expanduser('~/.cargo/bin')}:{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"

print("📥 2. Installing sigoden/aichat via Cargo...")
!cargo install aichat

print("⚡ 3. Installing argc CLI (required for llm-functions & MCP bridge)...")
# https://github.com/sigoden/argc
!curl -fsSL https://raw.githubusercontent.com/sigoden/argc/main/install.sh | sh -s -- --to /usr/local/bin

print("🔧 4. Setting up sigoden/llm-functions (MCP bridge support)...")
# https://github.com/sigoden/llm-functions
# https://github.com/sigoden/llm-functions/tree/main/mcp/bridge
functions_dir = os.path.expanduser("~/.config/aichat/functions")
os.makedirs(os.path.dirname(functions_dir), exist_ok=True)
!git clone https://github.com/sigoden/llm-functions {functions_dir} > /dev/null 2>&1 || (cd {functions_dir} && git pull)

print("🚀 5. Installing and launching Ollama background service (num_ctx=32768)...")
!curl -fsSL https://ollama.com/install.sh | sh

# Configure Ollama context length and start service in background
os.environ["OLLAMA_CONTEXT_LENGTH"] = "32768"
ollama_env = os.environ.copy()
subprocess.Popen(["ollama", "serve"], env=ollama_env)

# Health check: wait for Ollama daemon to respond
for _ in range(12):
    time.sleep(1)
    try:
        res = subprocess.run(["curl", "-s", "http://127.0.0.1:11434"], capture_output=True, text=True)
        if "Ollama is running" in res.stdout:
            print("   ✅ Ollama server is active (context length: 32768)!")
            break
    except Exception:
        pass

print("🧠 6. Pulling Ollama model (Gemma 4 QAT)...")
# Official Ollama Library Models:
# https://ollama.com/library/gemma4:12b-it-qat
# https://ollama.com/library/gemma4:e4b-it-qat
# https://ollama.com/library/gemma4:e2b-it-qat
!ollama pull gemma4:12b-it-qat
# --- Lightweight alternatives for Colab Free Tier / CPU ---
# !ollama pull gemma4:e4b-it-qat
# !ollama pull gemma4:e2b-it-qat

print("🛠️ 7. Installing Python dependencies, specify-cli (spec-kit) & spec-craft...")
!pip install -q uv pytest
!uv tool install specify-cli || uv tool install specify-cli --force --from git+https://github.com/github/spec-kit.git
!uv pip install --system git+https://github.com/akuroiwa/spec-craft.git

print("🎉 Core environment ready!")


In [ ]:
# ==============================================================================
# 2. Dynamic aichat Configuration (Auto-detect available Ollama models + Tool Calling)
# ==============================================================================
import os, subprocess, glob

config_dir = os.path.expanduser("~/.config/aichat")
os.makedirs(config_dir, exist_ok=True)

# Detect pulled models from Ollama
try:
    res = subprocess.run(["ollama", "list"], capture_output=True, text=True)
    lines = res.stdout.strip().split("\n")[1:]
    pulled_models = [line.split()[0] for line in lines if line]
except Exception:
    pulled_models = ["gemma4:12b-it-qat"]

if "gemma4:12b-it-qat" in pulled_models:
    default_model = "gemma4:12b-it-qat"
elif "gemma4:e4b-it-qat" in pulled_models:
    default_model = "gemma4:e4b-it-qat"
elif "gemma4:e2b-it-qat" in pulled_models:
    default_model = "gemma4:e2b-it-qat"
else:
    default_model = pulled_models[0] if pulled_models else "gemma4:12b-it-qat"

models_list = "\n".join([f"      - name: {m}" for m in pulled_models])

aichat_config = f"""
model: ollama:{default_model}
function_calling: true

clients:
  - type: openai-compatible
    name: ollama
    api_base: http://127.0.0.1:11434/v1
    models:
{models_list}
"""

with open(os.path.join(config_dir, "config.yaml"), "w", encoding="utf-8") as f:
    f.write(aichat_config.strip())

print(f"✅ aichat configured with active model: ollama:{default_model}")
print(f"   Detected available models: {pulled_models}")


## 1. Configure and Launch llm-functions MCP Bridge for Spec-Craft

`sigoden/llm-functions` allows external MCP tools to be bridged into `aichat` functions via `mcp.json` and `argc mcp start`.
Here we register `spec-craft` (`uv run spec-craft serve`) and start the bridge server.


In [ ]:
# ==============================================================================
# 3. Configure llm-functions MCP Bridge for spec-craft
# ==============================================================================
import json, os, subprocess

functions_dir = os.path.expanduser("~/.config/aichat/functions")
mcp_json_path = os.path.join(functions_dir, "mcp.json")

# Define MCP server configuration (same schema as Antigravity / Claude Desktop)
mcp_config = {
    "mcpServers": {
        "spec-craft": {
            "command": "uv",
            "args": [
                "run",
                "spec-craft",
                "serve"
            ]
        }
    }
}

with open(mcp_json_path, "w", encoding="utf-8") as f:
    json.dump(mcp_config, f, indent=2)

print(f"📝 Created {mcp_json_path}")

# Start the MCP bridge server and build tool binaries
# https://github.com/sigoden/llm-functions/tree/main/mcp/bridge
print("🚀 Starting argc MCP bridge server...")
!cd {functions_dir} && argc mcp start

# Check bridge logs
print("\n📋 argc MCP logs:")
!cd {functions_dir} && argc mcp logs || true

# Verify generated functions.json
functions_json_path = os.path.join(functions_dir, "functions.json")
if os.path.exists(functions_json_path):
    print(f"\n✅ functions.json created ({os.path.getsize(functions_json_path)} bytes)")
    !head -n 25 {functions_json_path}
else:
    print("⚠️ functions.json not found yet. Running argc build...")
    !cd {functions_dir} && argc build || true


> **Note (Troubleshooting): Emergency Ollama Server Restart**
> If the Ollama server ever becomes unresponsive or drops connections, execute the cell below to safely restart the service with `OLLAMA_CONTEXT_LENGTH=32768`.


In [ ]:
# Emergency restart for Ollama service
!pkill -9 -f ollama || true
!export OLLAMA_CONTEXT_LENGTH=32768 && ollama serve > /dev/null 2>&1 &
!sleep 2
!curl -s http://localhost:11434


## 2. Verify Prerequisites & Batch Execution Helper

Check the environment readiness using `spec-craft check-env` and load the robust artifact generator helper.


In [ ]:
# Verify environment readiness
!spec-craft check-env

# Test aichat non-streaming query execution (-S)
!aichat -S "Hello! Confirm in one short sentence that you are ready for Spec-Driven Development."


In [ ]:
# ==============================================================================
# Helper: Robust Batch Generator & Python Code Sanitizer
# ==============================================================================
import re, ast, os, subprocess

def clean_code_text(text: str) -> str:
    """
    Sanitize LLM output by removing reasoning tags (<think>...</think>),
    unescaping literal '\\n' sequences if returned as a single line,
    and stripping Markdown code fences.
    """
    # 1. Strip reasoning tags if present
    cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    
    # 2. Fix escaped newlines if output is packed into a single string literal
    if "\\n" in cleaned and "\n" not in cleaned:
        try:
            cleaned = cleaned.encode("utf-8").decode("unicode_escape")
        except Exception:
            cleaned = cleaned.replace("\\n", "\n")
            
    # 3. Extract code from ```python ... ``` block if wrapped
    match = re.search(r'```(?:python)?\s*(.*?)\s*```', cleaned, flags=re.DOTALL)
    if match:
        extracted = match.group(1).strip()
        if "\\n" in extracted and "\n" not in extracted:
            try:
                extracted = extracted.encode("utf-8").decode("unicode_escape")
            except Exception:
                extracted = extracted.replace("\\n", "\n")
        cleaned = extracted
        
    return cleaned

def generate_artifact(prompt: str, output_path: str, context_file: str = None, is_python: bool = False) -> str:
    """
    Execute aichat in non-streaming mode, clean reasoning (<think> tags) and
    Markdown code block wrappers, validate Python syntax (if is_python), and save to disk.
    """
    cmd = ["aichat", "-S"]
    if context_file and os.path.exists(context_file):
        cmd.extend(["-f", context_file])
    cmd.append(prompt)
    
    res = subprocess.run(cmd, capture_output=True, text=True)
    raw_output = res.stdout if res.returncode == 0 else (res.stdout + res.stderr)
    
    cleaned = clean_code_text(raw_output)
    
    if is_python:
        # Filter out stray markdown bullet/header lines if any slipped through
        lines = []
        for line in cleaned.splitlines():
            stripped = line.strip()
            # Retain comments, imports, functions, classes, and valid code lines
            if stripped.startswith(("#", "import ", "from ", "def ", "class ", "@", "if ", "try:", "except", "with ", "return", "pass", "raise", "assert", '"""', "'''")) or not stripped.startswith(("*", "-", "1.", "2.", "3.", "```")):
                lines.append(line)
        cleaned = "\n".join(lines).strip()
        
        # Validate Python AST syntax
        try:
            ast.parse(cleaned)
        except SyntaxError as e:
            print(f"⚠️ Warning: SyntaxError detected in generated code ({e}). Saving for inspection.")
    
    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(cleaned + "\n")
    
    print(f"📄 Saved: {output_path} ({len(cleaned.splitlines())} lines)")
    return cleaned

print("✅ Helper utility 'generate_artifact' loaded!")


## 3. Step 1: Draft the Strategic Specification in Obsidian (`obsidian/`)

In Spec-Craft, development begins with a strategic vision in Obsidian. We define the **What** and **Why** before diving into implementation details.

Let's prompt `aichat` in batch mode to draft a feature specification for a **Markdown Task Progress Analyzer CLI tool** (`obsidian/strategy_task_tracker.md`).


In [ ]:
import os
os.makedirs("obsidian", exist_ok=True)

# Draft Obsidian specification with YAML frontmatter and structured sections
prompt = """Create an Obsidian-compatible feature roadmap Markdown note with YAML frontmatter.
Title: 'Markdown Task Tracker CLI'.
Include YAML frontmatter with title, tags: [sdd, cli, utility], and status: draft.
Include H1 sections: '# 1. Overview', '# 2. User Stories', '# 3. Technical Requirements'.
The tool parses a markdown string or file, finds all '- [ ]' (pending) and '- [x]' (completed) checklist items, and calculates total, completed, pending, and completion percentage."""

content = generate_artifact(prompt, "obsidian/strategy_task_tracker.md")

# Display the generated Obsidian strategic specification
!cat obsidian/strategy_task_tracker.md


## 4. Step 2: Inspect and Parse the Strategic Roadmap with Spec-Craft

Spec-Craft provides structured parsers to inspect Obsidian storyboards, extracting metadata and sections safely without losing context.


In [ ]:
from pathlib import Path
from spec_craft.core.parser import StoryboardParser

# Parse the generated storyboard note
parser = StoryboardParser(Path("obsidian"))
storyboard = parser.parse("strategy_task_tracker.md")

print(f"📄 File Path: {storyboard.file_path}")
print(f"🏷️  Metadata: {storyboard.metadata}")
print(f"📑 Sections Found ({len(storyboard.sections)}):")
for section in storyboard.sections:
    print(f"   [Level {section.level}] {section.title}")


## 5. Step 3: Automated Tactical Implementation (Python Module)

Translate the Strategy specification into executable Python code (`task_tracker.py`).


In [ ]:
# Generate the Python implementation based on the Obsidian spec
prompt_impl = """Implement the complete Python module 'task_tracker.py'.
Provide a class 'TaskTracker' and a function 'analyze_tasks(content: str) -> dict'
returning {'total': int, 'completed': int, 'pending': int, 'percentage': float}.
Output ONLY raw Python code without Markdown backticks."""

generate_artifact(prompt_impl, "task_tracker.py", context_file="obsidian/strategy_task_tracker.md", is_python=True)
print("✨ Generated and validated task_tracker.py")


## 6. Step 4: Automated Test Suite Generation (`test_task_tracker.py`)

Generate a concise pytest test suite (`test_task_tracker.py`) covering the core requirements without triggering infinite reasoning loops.


In [ ]:
# Generate focused unit tests with pytest (limited to 4 target cases)
prompt_test = """Write a concise pytest test suite in 'test_task_tracker.py' with exactly 4 unit tests:
1. test_empty_content: empty text returns total 0 and percentage 0.0
2. test_all_pending: all '- [ ]' items returns completed 0, pending = total
3. test_all_completed: all '- [x]' items returns pending 0, percentage 100.0
4. test_mixed_tasks: mixed '- [ ]' and '- [x]' items returns correct counts and percentage

Use only the public API: analyze_tasks(content: str) and TaskTracker.
Output ONLY raw Python code without explanations or markdown wrappers."""

generate_artifact(prompt_test, "test_task_tracker.py", context_file="task_tracker.py", is_python=True)
print("✨ Generated and validated test_task_tracker.py")


> **Tip (Manual Override Fallback)**:
> If you ever want to write or edit `test_task_tracker.py` manually in Colab, you can use the `%%writefile` cell magic:
> ```python
> %%writefile test_task_tracker.py
> import pytest
> from task_tracker import analyze_tasks, TaskTracker
> 
> def test_empty_tasks():
>     res = analyze_tasks("")
>     assert res["total"] == 0
>     assert res["percentage"] == 0.0
> ```


## 7. Step 5: Verification and Test Execution

Run `pytest` to verify that the generated implementation passes all acceptance criteria.


In [ ]:
# Execute unit tests
!pytest test_task_tracker.py -v


## Summary & Next Steps

Congratulations! You have completed the automated Spec-Driven Development cycle in Google Colab:
- **Strategy (Obsidian)**: Captured vision and user requirements.
- **Spec-Craft**: Parsed structure and validated roadmap.
- **Tactics (Python & pytest)**: Generated clean, test-verified implementation in distinct, reproducible steps.

### Interactive Development with OpenCode
To experience interactive agent-driven development using native MCP tool calling and live TUI, open the companion notebook:
- **[OpenCode MCP Interactive Notebook](https://colab.research.google.com/github/akuroiwa/spec-craft/blob/main/docs/source/notebooks/spec_craft_opencode.ipynb)**

### Resources
- **GitHub Repository**: [https://github.com/akuroiwa/spec-craft](https://github.com/akuroiwa/spec-craft)
- **Documentation**: [https://spec-craft.readthedocs.io/](https://spec-craft.readthedocs.io/)
- **Zenn Book / Articles**: [spec-craft Development Notes](https://zenn.dev/akuroiwa/books/dev-note-spec-craft)
